# Stock 1 Within-Time-ID Bucket Model

Uses `optiver_aggregated.csv`, filters to `stock_id == 1`, trains on buckets 1-16 within each `time_id`, and predicts buckets 17-20.


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold

DATA_PATH = '/Users/rosakwak/Desktop/DATA3888/DATA3888G08/optiver_aggregated.csv'
OUTPUT_DIR = '/Users/rosakwak/Desktop/DATA3888/DATA3888G08/stock1_outputs'
TARGET_STOCK_ID = 1
N_SPLITS = 5

df = pd.read_csv(DATA_PATH)

required_cols = ['stock_id', 'time_id', 'time_bucket', 'BidAskSpread_mean', 'RV']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

df = df[required_cols].copy()
df['time_bucket'] = df['time_bucket'].astype(int)
df = df[df['stock_id'] == TARGET_STOCK_ID].copy()
df = df.sort_values(['stock_id', 'time_id', 'time_bucket']).reset_index(drop=True)

if df.empty:
    raise ValueError(f'No rows found for stock_id={TARGET_STOCK_ID}')

print('Target stock_id:', TARGET_STOCK_ID)
print('Data shape:', df.shape)
print('time_bucket range:', df['time_bucket'].min(), 'to', df['time_bucket'].max())
print(df.head())


def qlike_loss(y_true, y_pred, eps=1e-12):
    y_true = np.maximum(np.asarray(y_true), eps)
    y_pred = np.maximum(np.asarray(y_pred), eps)
    return np.mean(y_true / y_pred - np.log(y_true / y_pred) - 1)


def add_within_time_id_features(data):
    df_feat = data.sort_values(['stock_id', 'time_id', 'time_bucket']).copy()
    g = df_feat.groupby(['stock_id', 'time_id'], sort=False)

    df_feat['RV_lag1'] = g['RV'].shift(1)
    df_feat['RV_mean5'] = g['RV'].transform(
        lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
    )
    df_feat['RV_mean10'] = g['RV'].transform(
        lambda s: s.shift(1).rolling(window=10, min_periods=1).mean()
    )

    return df_feat


def recursive_predict_last_buckets(data, model, feature_cols, train_end_bucket=16, pred_buckets=(17, 18, 19, 20)):
    wide = data.pivot_table(
        index=['stock_id', 'time_id'],
        columns='time_bucket',
        values='RV',
        aggfunc='first'
    )
    wide = wide.reindex(columns=range(1, max(pred_buckets) + 1))

    history = [wide[bucket].astype(float) for bucket in range(1, train_end_bucket + 1)]
    prediction_frames = []

    for bucket in pred_buckets:
        feature_frame = pd.DataFrame({
            'time_bucket': bucket,
            'RV_lag1': history[-1],
            'RV_mean5': pd.concat(history[-5:], axis=1).mean(axis=1),
            'RV_mean10': pd.concat(history[-10:], axis=1).mean(axis=1)
        }, index=wide.index)

        valid_features = feature_frame[feature_cols].notna().all(axis=1)
        y_pred = pd.Series(np.nan, index=wide.index, dtype=float)
        y_pred.loc[valid_features] = np.maximum(
            model.predict(feature_frame.loc[valid_features, feature_cols]),
            1e-12
        )

        actual = wide[bucket]
        valid_predictions = actual.notna() & y_pred.notna()

        bucket_predictions = feature_frame.loc[valid_predictions, []].reset_index()
        bucket_predictions['time_bucket'] = bucket
        bucket_predictions['RV'] = actual.loc[valid_predictions].to_numpy()
        bucket_predictions['pred_RV'] = y_pred.loc[valid_predictions].to_numpy()
        prediction_frames.append(bucket_predictions)

        history.append(y_pred)

    return pd.concat(prediction_frames, ignore_index=True)


def evaluate_model(y_true, y_pred):
    y_pred = np.maximum(np.asarray(y_pred), 1e-12)

    return {
        'MSE': mean_squared_error(y_true, y_pred),
        'QLIKE': qlike_loss(y_true, y_pred),
        'RV_variance': np.var(y_true),
        'pred_RV_variance': np.var(y_pred)
    }


TRAIN_END_BUCKET = 16
PRED_BUCKETS = (17, 18, 19, 20)
FEATURE_COLS = ['time_bucket', 'RV_lag1', 'RV_mean5', 'RV_mean10']

df_feat = add_within_time_id_features(df)

train_all = df_feat[df_feat['time_bucket'].between(2, TRAIN_END_BUCKET)].dropna(
    subset=['RV'] + FEATURE_COLS
).copy()

groups = train_all['time_id']
n_time_ids = groups.nunique()
n_splits = min(N_SPLITS, n_time_ids)
if n_splits < 2:
    raise ValueError('At least 2 unique time_id values are required for cross-validation')

group_kfold = GroupKFold(n_splits=n_splits)
fold_predictions = []

for fold, (train_idx, test_idx) in enumerate(
    group_kfold.split(train_all, train_all['RV'], groups=groups),
    start=1
):
    fold_train = train_all.iloc[train_idx].copy()
    test_time_ids = train_all.iloc[test_idx]['time_id'].unique()
    fold_test_data = df[df['time_id'].isin(test_time_ids)].copy()

    model = LinearRegression()
    model.fit(fold_train[FEATURE_COLS], fold_train['RV'])

    fold_pred = recursive_predict_last_buckets(
        fold_test_data,
        model,
        FEATURE_COLS,
        train_end_bucket=TRAIN_END_BUCKET,
        pred_buckets=PRED_BUCKETS
    )
    fold_pred['fold'] = fold
    fold_predictions.append(fold_pred)

predictions = pd.concat(fold_predictions, ignore_index=True)

results = (
    predictions
    .groupby(['fold', 'stock_id', 'time_id'])
    .agg(
        target_RV=('RV', 'mean'),
        pred_RV=('pred_RV', 'mean'),
        n_predicted_buckets=('time_bucket', 'count')
    )
    .reset_index()
)

mean_spread = (
    df[df['time_bucket'] <= TRAIN_END_BUCKET]
    .groupby(['stock_id', 'time_id'])['BidAskSpread_mean']
    .mean()
    .reset_index()
    .rename(columns={'BidAskSpread_mean': 'mean_spread'})
)
results = results.merge(mean_spread, on=['stock_id', 'time_id'], how='left')
results['QLIKE'] = (
    np.maximum(results['target_RV'].to_numpy(), 1e-12)
    / np.maximum(results['pred_RV'].to_numpy(), 1e-12)
    - np.log(
        np.maximum(results['target_RV'].to_numpy(), 1e-12)
        / np.maximum(results['pred_RV'].to_numpy(), 1e-12)
    )
    - 1
)
results['MSE'] = (results['pred_RV'] - results['target_RV']) ** 2

time_id_metrics = []
for _, row in results.iterrows():
    metrics = {
        'fold': row['fold'],
        'time_id': row['time_id'],
        'num_predicted_buckets': row['n_predicted_buckets'],
        'MSE': row['MSE'],
        'QLIKE': row['QLIKE'],
        'target_RV': row['target_RV'],
        'pred_RV': row['pred_RV'],
        'mean_spread': row['mean_spread']
    }
    time_id_metrics.append(metrics)

time_id_metrics_table = pd.DataFrame(time_id_metrics)
time_id_metrics_table = time_id_metrics_table[
    ['fold', 'time_id', 'num_predicted_buckets', 'target_RV', 'pred_RV', 'mean_spread', 'MSE', 'QLIKE']
].sort_values(['fold', 'time_id']).reset_index(drop=True)

fold_metrics = []
for fold, fold_df in results.groupby('fold'):
    fold_metrics.append({
        'fold': fold,
        'num_time_ids': fold_df['time_id'].nunique(),
        'mean_QLIKE': fold_df['QLIKE'].mean(),
        'median_QLIKE': fold_df['QLIKE'].median(),
        'mean_MSE': fold_df['MSE'].mean(),
        'median_MSE': fold_df['MSE'].median(),
        'mean_spread': fold_df['mean_spread'].mean()
    })

fold_metrics_table = pd.DataFrame(fold_metrics)
fold_metrics_table = fold_metrics_table[
    ['fold', 'num_time_ids', 'mean_QLIKE', 'median_QLIKE', 'mean_MSE', 'median_MSE', 'mean_spread']
].sort_values('fold').reset_index(drop=True)

per_stock = (
    results.groupby('stock_id')
    .agg(
        mean_QLIKE=('QLIKE', 'mean'),
        median_QLIKE=('QLIKE', 'median'),
        mean_MSE=('MSE', 'mean'),
        median_MSE=('MSE', 'median'),
        mean_spread=('mean_spread', 'mean'),
        n_sessions=('time_id', 'count')
    )
    .reset_index()
    .sort_values('mean_QLIKE')
)

per_timeid = (
    results.groupby('time_id')
    .agg(
        mean_QLIKE=('QLIKE', 'mean'),
        mean_MSE=('MSE', 'mean'),
        target_RV=('target_RV', 'mean'),
        pred_RV=('pred_RV', 'mean'),
        mean_spread=('mean_spread', 'mean')
    )
    .reset_index()
)

overall_metrics_table = pd.DataFrame([{
    'mean_time_id_QLIKE': time_id_metrics_table['QLIKE'].mean(),
    'median_time_id_QLIKE': time_id_metrics_table['QLIKE'].median(),
    'mean_time_id_MSE': time_id_metrics_table['MSE'].mean(),
    'median_time_id_MSE': time_id_metrics_table['MSE'].median(),
    'mean_fold_QLIKE': fold_metrics_table['mean_QLIKE'].mean(),
    'mean_fold_MSE': fold_metrics_table['mean_MSE'].mean(),
    'num_time_ids': time_id_metrics_table['time_id'].nunique()
}])

print('CV folds:', n_splits)
print('Training candidate rows:', len(train_all))
print('Prediction rows:', len(predictions))
print('Fold metrics:')
print(fold_metrics_table)
print('Overall evaluation summary:')
print(overall_metrics_table)
print('Per-time_id metrics:')
print(time_id_metrics_table)

os.makedirs(OUTPUT_DIR, exist_ok=True)

eval_results_path = os.path.join(OUTPUT_DIR, 'stock1_eval_results.csv')
per_stock_path = os.path.join(OUTPUT_DIR, 'stock1_per_stock.csv')
per_timeid_path = os.path.join(OUTPUT_DIR, 'stock1_per_timeid.csv')
fold_metrics_path = os.path.join(OUTPUT_DIR, 'stock1_fold_metrics.csv')
overall_metrics_path = os.path.join(OUTPUT_DIR, 'stock1_overall_metrics.csv')
bucket_predictions_path = os.path.join(OUTPUT_DIR, 'stock1_bucket_predictions.csv')

results[['stock_id', 'time_id', 'fold', 'mean_spread', 'pred_RV', 'target_RV', 'QLIKE', 'MSE']].to_csv(
    eval_results_path,
    index=False
)
per_stock.to_csv(per_stock_path, index=False)
per_timeid.to_csv(per_timeid_path, index=False)
fold_metrics_table.to_csv(fold_metrics_path, index=False)
overall_metrics_table.to_csv(overall_metrics_path, index=False)
predictions.to_csv(bucket_predictions_path, index=False)

print('Saved eval results to:', eval_results_path)
print('Saved per-stock summary to:', per_stock_path)
print('Saved per-time_id summary to:', per_timeid_path)
print('Saved fold metrics to:', fold_metrics_path)
print('Saved overall metrics to:', overall_metrics_path)
print('Saved bucket predictions to:', bucket_predictions_path)

print('\n-- Evaluation Summary (stock_id=1, out-of-sample) --')
print(f"  Total sessions : {len(results):,}")
print(f"  Median QLIKE   : {results['QLIKE'].median():.6f}")
print(f"  Mean   QLIKE   : {results['QLIKE'].mean():.6f}")
print(f"  Median MSE     : {results['MSE'].median():.2e}")
print(f"  Mean   MSE     : {results['MSE'].mean():.2e}")
